In [ ]:
# Cell 92 (00_precheck, cell 1): CONFIG + split asserts + inspect locked preprocessing (READS ONLY; writes nothing)
import os, sys, json, pickle
import numpy as np, pandas as pd

assert os.environ.get('PYTHONHASHSEED') == '0', "PYTHONHASHSEED is not '0': wrong kernel, STOP"
SEED = 42
PREFIX = 'sel_'
ROOT = os.environ["CREDIT_BN_ROOT"]
PROC = ROOT + "/Data/processed/"
FVM = ROOT + "/full_variable/models/"
sys.path.insert(0, ROOT + "/shared/src")

L = lambda p: pickle.load(open(p, 'rb'))
split = L(PROC + "split_indices.pkl")
train_idx, test_idx = np.asarray(split['train_idx']), np.asarray(split['test_idx'])
y_train, y_test = np.asarray(split['y_train']), np.asarray(split['y_test'])
assert len(train_idx) == 119999 and len(test_idx) == 30000
assert len(y_train) == 119999 and len(y_test) == 30000
assert round(100 * y_train.mean(), 2) == 6.68 and round(100 * y_test.mean(), 2) == 6.68
assert len(set(train_idx) & set(test_idx)) == 0
print(f"split OK: train {len(train_idx):,} ({y_train.mean():.4%}), test {len(test_idx):,} ({y_test.mean():.4%})")

no = json.load(open(PROC + "node_orders.json"))
print("\nKEEP_VARS:", no['KEEP_VARS'])
print("DROP_VARS:", no['DROP_VARS'])

print("\n--- df_clean (raw cleaned, 149,999 rows) ---")
dfc = L(PROC + "df_clean.pkl")
print(dfc.shape, "index range", dfc.index.min(), dfc.index.max())
print(pd.DataFrame({'dtype': dfc.dtypes.astype(str), 'nan': dfc.isna().sum(), 'min': dfc.min(numeric_only=True), 'max': dfc.max(numeric_only=True)}))
print("split indices are labels in df_clean:", bool(np.isin(train_idx, dfc.index).all() and np.isin(test_idx, dfc.index).all()))

print("\n--- df_train_lr_rf (input to the locked imputer/encoder) ---")
dtr = L(PROC + "df_train_lr_rf.pkl")
print(dtr.shape, list(dtr.columns))
print("NaN per column:", dtr.isna().sum()[dtr.isna().sum() > 0].to_dict())
print("index equals train_idx:", bool(np.array_equal(dtr.index.values, train_idx)))

print("\n--- locked imputer ---")
imp = L(FVM + "imputer.pkl")
print(imp)
print("params:", {k: (v if isinstance(v, (int, float, str, bool, type(None))) else repr(v)[:80]) for k, v in imp.get_params().items()})
print("fitted on columns:", list(getattr(imp, 'feature_names_in_', [])) or f"n_features_in_={imp.n_features_in_}")

print("\n--- locked encoder ---")
enc = L(FVM + "encoder.pkl")
print({k: (v if isinstance(v, (int, float, str, bool, type(None))) else repr(v)[:80]) for k, v in enc.get_params().items()})
print("input columns:", list(getattr(enc, 'feature_names_in_', [])))
for c, cats in zip(getattr(enc, 'feature_names_in_', range(len(enc.categories_))), enc.categories_):
    print(f"  {c}: {list(cats)}")

print("\n--- locked design matrix X_train_df (35 cols) ---")
Xtr = L(PROC + "X_train_df.pkl")
print(Xtr.shape, list(Xtr.columns))

print("\n--- locked LR ---")
lr = L(FVM + "lr_final.pkl")
print({k: v for k, v in lr.get_params().items() if k in ('C', 'penalty', 'solver', 'class_weight', 'max_iter', 'random_state', 'tol')})

In [ ]:
# Cell 93 (00_precheck, cell 2): READ-ONLY -- how the locked pipeline went imputed values -> bins -> one-hot
import re
hist = open(ROOT + "/archive/kernel/In_history_at_cell78.txt").read()
parts = re.split(r"# ===== In\[(\d+)\] =====\n", hist)
blocks = {int(parts[i]): parts[i + 1] for i in range(1, len(parts) - 1, 2)}
KEYS = ['IterativeImputer(', 'expm1', 'OneHotEncoder(', 'bin_natural_category(', 'df_train_lr_rf =', 'df_test_lr_rf =', 'income_edges']

def active(c):  # lines that are not comments
    return [l for l in c.splitlines() if l.strip() and not l.lstrip().startswith('#')]

hits = [i for i, c in sorted(blocks.items()) if i < 74 and any(k in "\n".join(active(c)) for k in KEYS)]
print("history cells with ACTIVE lines containing the keywords:", hits)
for i in hits:
    lines = blocks[i].splitlines()
    print(f"\n===== In[{i}] ({len(lines)} lines; showing up to 70) =====")
    print("\n".join(lines[:70]))

print("\n===== recovered source: bin_natural_category =====")
rec = open(ROOT + "/shared/src/_recovered_from_kernel.py").read()
m = re.search(r"# ---- bin_natural_category ----\n(.*?)(?=\n# ---- )", rec, re.S)
print(m.group(1) if m else "not found")

print("\n===== saved bin edges =====")
pa = L(PROC + "preprocessing_artifacts.pkl")
for k, v in pa['bin_edges'].items():
    print(" ", k, [round(float(x), 4) for x in v])
for k in ['income_edges', 'income_edges_adj']:
    print(" ", k, [round(float(x), 4) for x in pa[k]])

print("\n===== df_train_imputed (continuous values after imputation) =====")
dti = L(PROC + "df_train_imputed.pkl")
print(dti.shape, list(dti.columns))
print(dti.describe().T[['min', 'max']])
for c in ['Number of Dependents', 'Real Estate Loans', 'Open Credit Lines']:
    if c in dti.columns:
        print(f"  {c}: non-integer values = {int((dti[c] % 1 != 0).sum())}")

In [ ]:
# Cell 94 (00_precheck, cell 3): rebuild locked preprocessing as a function, VALIDATE against locked X_train_df/X_test_df (READS ONLY; writes nothing)
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import OneHotEncoder

dfc = L(PROC + "df_clean.pkl")
EDGES = L(PROC + "preprocessing_artifacts.pkl")['bin_edges']

# locked orders (from the recovered history)
IMPUTE_ORDER = ['Credit Utilisation Rate', 'Age', 'Debt Ratio', 'Monthly Income', 'Open Credit Lines',
                '30-59 Days Past Due', '90+ Days Late', '60-89 Days Past Due', 'Number of Dependents', 'Real Estate Loans']
DESIGN_ORDER = ['Credit Utilisation Rate', 'Age', 'Debt Ratio', 'Open Credit Lines', '30-59 Days Past Due',
                '90+ Days Late', '60-89 Days Past Due', 'Real Estate Loans', 'Monthly Income', 'Number of Dependents']
EDGE_VARS = ['Credit Utilisation Rate', 'Age', 'Debt Ratio', 'Open Credit Lines', 'Monthly Income']   # cut with saved edges
NAT_VARS = ['30-59 Days Past Due', '90+ Days Late', '60-89 Days Past Due', 'Number of Dependents', 'Real Estate Loans']

def bin_natural_category(series, max_category=3):   # EXACT copy of the locked function (note: int(x) truncation)
    labels = [str(i) for i in range(max_category)] + [f"{max_category}+"]
    def map_val(x):
        if pd.isnull(x):
            return np.nan
        return labels[int(x)] if x < max_category else labels[-1]
    return pd.Categorical(series.map(map_val), categories=labels, ordered=True)

def cut_edges(s, e):
    ea = np.array(e, float).copy(); ea[0] = -np.inf; ea[-1] = np.inf
    return pd.cut(s, bins=ea, include_lowest=True)

def build_design(raw_cols):
    """Locked preprocessing, refit on TRAIN ONLY for the given raw columns."""
    imp_cols = [c for c in IMPUTE_ORDER if c in raw_cols]
    des_cols = [c for c in DESIGN_ORDER if c in raw_cols]
    assert set(imp_cols) == set(raw_cols) == set(des_cols)
    tr, te = dfc.loc[train_idx, imp_cols].copy(), dfc.loc[test_idx, imp_cols].copy()
    if 'Monthly Income' in imp_cols:
        tr['Monthly Income'], te['Monthly Income'] = np.log1p(tr['Monthly Income']), np.log1p(te['Monthly Income'])
    imp = None
    if tr.isna().any().any() or te.isna().any().any():
        imp = IterativeImputer(random_state=42).fit(tr)
        tr = pd.DataFrame(imp.transform(tr), columns=imp_cols, index=tr.index)
        te = pd.DataFrame(imp.transform(te), columns=imp_cols, index=te.index)
    if 'Monthly Income' in imp_cols:
        tr['Monthly Income'], te['Monthly Income'] = np.expm1(tr['Monthly Income']).clip(lower=0), np.expm1(te['Monthly Income']).clip(lower=0)
    def binned(df):
        out = pd.DataFrame(index=df.index)
        for c in des_cols:
            out[c + ' (bin)'] = cut_edges(df[c], EDGES[c]) if c in EDGE_VARS else bin_natural_category(df[c], 3)
        return out.astype(str)
    s_tr, s_te = binned(tr), binned(te)
    enc = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='error').fit(s_tr)
    names = enc.get_feature_names_out(s_tr.columns)
    return dict(Xtr=pd.DataFrame(enc.transform(s_tr), columns=names, index=tr.index),
                Xte=pd.DataFrame(enc.transform(s_te), columns=names, index=te.index),
                s_tr=s_tr, s_te=s_te, imputer=imp, encoder=enc, imp_tr=tr, imp_te=te)

# ---- validation: all 10 columns must reproduce the locked matrices ----
full = build_design(IMPUTE_ORDER)
Xtr_L, Xte_L = L(PROC + "X_train_df.pkl"), L(PROC + "X_test_df.pkl")
ok_cols = list(full['Xtr'].columns) == list(Xtr_L.columns)
ok_idx = full['Xtr'].index.equals(Xtr_L.index) and full['Xte'].index.equals(Xte_L.index)
ok_tr = ok_cols and np.array_equal(full['Xtr'].values, Xtr_L.values)
ok_te = ok_cols and np.array_equal(full['Xte'].values, Xte_L.values)
print(f"columns equal: {ok_cols} | index equal: {ok_idx} | X_train identical: {ok_tr} | X_test identical: {ok_te}")
if not (ok_tr and ok_te):
    d = (full['Xtr'].values != Xtr_L.values).sum(axis=0) if ok_cols else None
    if d is not None:
        print("train mismatching cells by column:", {c: int(n) for c, n in zip(Xtr_L.columns, d) if n})
    print("-> STOP: rebuilt pipeline does not reproduce the locked matrices.")
s_L = L(PROC + "df_train_lr_rf.pkl").astype(str)
print("string frame equals locked df_train_lr_rf:", bool((full['s_tr'].values == s_L.values).all()))

# ---- quirk diagnostic: imputed Number of Dependents under the locked int() rule ----
for nm, imp_df, idx in [('train', full['imp_tr'], train_idx), ('test', full['imp_te'], test_idx)]:
    miss = dfc.loc[idx, 'Number of Dependents'].isna().values
    v = imp_df.loc[miss, 'Number of Dependents']
    lab = full['s_tr' if nm == 'train' else 's_te'].loc[miss, 'Number of Dependents (bin)']
    print(f"\n{nm}: imputed dependents rows = {int(miss.sum())}; min={v.min():.3f} max={v.max():.3f}")
    print(f"  value <= -1 (int() gives -1 -> labelled '3+'): {int((v <= -1).sum())}")
    print(f"  -1 < value < 0 (labelled '0'): {int(((v > -1) & (v < 0)).sum())}")
    print("  labels assigned:", lab.value_counts().sort_index().to_dict())

In [ ]:
# Cell 95 (00_precheck, cell 4): VALIDATE LR refit on rebuilt full matrix vs locked LR (READS ONLY; writes nothing)
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

lr_locked = L(FVM + "lr_final.pkl")
abl = L(ROOT + "/Data/GiveMeSomeCredit/ablation_predictions.pkl")
lr_new = clone(lr_locked).fit(full['Xtr'].values, y_train)
p_new = lr_new.predict_proba(full['Xte'].values)[:, 1]
p_locked_obj = lr_locked.predict_proba(Xte_L.values)[:, 1]
p_locked_file = np.asarray(abl['LR_all'], float)

print("params:", {k: v for k, v in lr_new.get_params().items() if k in ('C', 'penalty', 'solver', 'class_weight', 'max_iter', 'random_state', 'tol')})
print(f"AUC refit        : {roc_auc_score(y_test, p_new):.6f}")
print(f"AUC locked object: {roc_auc_score(y_test, p_locked_obj):.6f}")
print(f"AUC locked file  : {roc_auc_score(y_test, p_locked_file):.6f}   (expected 0.8618)")
print(f"max|refit - locked object| = {np.max(np.abs(p_new - p_locked_obj)):.3g}")
print(f"max|refit - locked file|   = {np.max(np.abs(p_new - p_locked_file)):.3g}")
print(f"max|coef diff|             = {np.max(np.abs(lr_new.coef_ - lr_locked.coef_)):.3g}")
print("PASS" if round(roc_auc_score(y_test, p_new), 4) == 0.8618 else "CHECK: AUC differs from 0.8618")

In [ ]:
# Cell 96 (00_precheck, cell 5): Spec A / Spec B build, LR (locked params, no tuning), metrics, paired bootstrap, SAVE (sel_). STOPS after the table.
# WRITES (new, via safe_save): selection/tables/sel_precheck_table.csv, sel_precheck_bootstrap.csv,
#                              selection/predictions/sel_precheck_predictions.pkl, selection/tables/sel_precheck_notes.json
import warnings, time, platform
warnings.filterwarnings("ignore")
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import ks_2samp
import sklearn, pgmpy
from safe_save import safe_save

DROP = no['DROP_VARS']                                   # e.g. 'Credit Utilisation Rate (bin)'
DROP_RAW = [d.replace(' (bin)', '') for d in DROP]
SPEC = {'A': ['Age', 'Debt Ratio', 'Monthly Income', 'Open Credit Lines', 'Real Estate Loans', 'Number of Dependents'],
        'B': ['Age', 'Debt Ratio', 'Open Credit Lines', 'Real Estate Loans']}
# the spec A list must equal KEEP_VARS from node_orders.json
assert sorted(SPEC['A']) == sorted(v.replace(' (bin)', '') for v in no['KEEP_VARS'])

lr_locked = L(FVM + "lr_final.pkl")
abl = L(ROOT + "/Data/GiveMeSomeCredit/ablation_predictions.pkl")
p_locked = np.asarray(abl['LR_all'], float)             # locked full-variable LR test probabilities (read-only)

D, P = {}, {}
for k, cols in SPEC.items():
    d = build_design(cols)                               # refit on TRAIN ONLY, only this spec's columns
    D[k] = d
    # --- no dropped variable anywhere ---
    enc_in = list(d['encoder'].feature_names_in_)
    assert not any(x in enc_in for x in DROP), f"dropped var in encoder input ({k})"
    assert not any(any(x in n for x in DROP) for n in d['Xtr'].columns), f"dropped var in design columns ({k})"
    if d['imputer'] is not None:
        imp_in = list(d['imputer'].feature_names_in_)
        assert not any(x in imp_in for x in DROP_RAW), f"dropped var in imputer ({k})"
        print(f"Spec {k}: imputer fitted on {imp_in}")
    else:
        print(f"Spec {k}: no missing values -> no imputer fitted (asserting zero NaN)")
        assert not dfc.loc[train_idx, cols].isna().any().any() and not dfc.loc[test_idx, cols].isna().any().any()
    print(f"Spec {k}: design {d['Xtr'].shape} train, {d['Xte'].shape} test; no dropped column in imputer/encoder/design: OK")
    lr = clone(lr_locked).fit(d['Xtr'].values, y_train)  # same params as locked, class_weight balanced, NO tuning
    P[k] = lr.predict_proba(d['Xte'].values)[:, 1]
    assert len(P[k]) == 30000

# --- Note: imputed Dependents counts in Spec A (read-only; no branching) ---
dep = {}
for nm, imp_df, idx, s in [('train', D['A']['imp_tr'], train_idx, D['A']['s_tr']), ('test', D['A']['imp_te'], test_idx, D['A']['s_te'])]:
    miss = dfc.loc[idx, 'Number of Dependents'].isna().values
    v = imp_df.loc[miss, 'Number of Dependents']
    lab = s.loc[miss, 'Number of Dependents (bin)'].value_counts().sort_index().to_dict()
    dep[nm] = dict(n_imputed=int(miss.sum()), min=float(v.min()), max=float(v.max()), lt0=int((v < 0).sum()),
                   le_minus1=int((v <= -1).sum()), labels=lab)
    print(f"Spec A {nm}: imputed Dependents rows={dep[nm]['n_imputed']} min={dep[nm]['min']:.3f} max={dep[nm]['max']:.3f} | "
          f"<0: {dep[nm]['lt0']} | <=-1 (wrap-around label): {dep[nm]['le_minus1']} | labels {lab}")

# --- metrics ---
def metrics(p):
    a = roc_auc_score(y_test, p)
    return dict(AUC=a, Gini=2 * a - 1, KS=ks_2samp(p[y_test == 1], p[y_test == 0]).statistic, AP=average_precision_score(y_test, p))
rows = {'Spec A (6 vars)': metrics(P['A']), 'Spec B (4 vars: no Income/Dependents)': metrics(P['B']), 'Locked full-variable LR': metrics(p_locked)}
tbl = pd.DataFrame(rows).T
assert round(tbl.loc['Locked full-variable LR', 'AUC'], 4) == 0.8618

# --- paired bootstrap: same resamples for every comparison (2,000, seed 42) ---
rng = np.random.RandomState(SEED)
n = len(y_test)
auc_b = {k: np.empty(2000) for k in ['A', 'B', 'L']}
pp = {'A': P['A'], 'B': P['B'], 'L': p_locked}
t0 = time.time()
for i in range(2000):
    ix = rng.randint(0, n, n)
    yy = y_test[ix]
    if yy.min() == yy.max():
        raise RuntimeError("degenerate resample")
    for k in auc_b:
        auc_b[k][i] = roc_auc_score(yy, pp[k][ix])
print(f"bootstrap done in {time.time() - t0:.0f}s")
obs = {k: roc_auc_score(y_test, pp[k]) for k in pp}
def comp(label, a, b):
    d = auc_b[a] - auc_b[b]
    lo, hi = np.percentile(d, [2.5, 97.5])
    p = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
    p = max(p, 1 / 2001)                                 # floor: resolution limit of 2,000 resamples
    return dict(comparison=label, diff=obs[a] - obs[b], ci_lo=lo, ci_hi=hi, p_value=p)
boot = pd.DataFrame([comp('A - B', 'A', 'B'), comp('A - locked LR', 'A', 'L'), comp('B - locked LR', 'B', 'L')])

pd.options.display.float_format = '{:.4f}'.format
print("\n=== PHASE 1 TABLE: test set, n = 30,000, LR with locked hyperparameters (C=100, l2, saga, balanced), no tuning ===")
print(tbl.to_string())
print("\n=== Paired bootstrap of AUC differences (2,000 resamples, seed 42; percentile 95% CI; two-sided p, floor 0.0005) ===")
print(boot.to_string(index=False))
print("\nNote: Spec A/B imputer and encoder refit on TRAIN ONLY using only that spec's columns. Locked LR = full-variable, read-only, reference AUC 0.8618.")

# --- save (sel_ prefix, safe_save refuses overwrite and protected folders) ---
notes = dict(spec=SPEC, imputed_dependents_specA=dep, seed=SEED, n_bootstrap=2000, n_test=int(n),
             lr_params={k: v for k, v in lr_locked.get_params().items() if k in ('C', 'penalty', 'solver', 'class_weight', 'max_iter', 'random_state', 'tol')},
             versions=dict(python=platform.python_version(), sklearn=sklearn.__version__, pgmpy=pgmpy.__version__, numpy=np.__version__, pandas=pd.__version__),
             PYTHONHASHSEED=os.environ.get('PYTHONHASHSEED'), timestamp=time.strftime('%Y-%m-%d %H:%M'),
             note="Phase 1 pre-check, not variable selection. Locked binning replicated exactly (int() rule, no clipping).")
S = ROOT + "/selection/"
for obj, rel, kw in [(tbl, "tables/sel_precheck_table.csv", {}), (boot, "tables/sel_precheck_bootstrap.csv", {"index": False}),
                     ({'A': P['A'], 'B': P['B'], 'test_idx': test_idx}, "predictions/sel_precheck_predictions.pkl", {}),
                     (notes, "tables/sel_precheck_notes.json", {})]:
    path, h = safe_save(obj, S + rel, **kw)
    print("saved:", rel, h[:12])